# Turkish LLM Reasoning Benchmark — Dataset Overview

This notebook provides an overview of the datasets used in the Turkish LLM reasoning project.

The project contains two main datasets:

- **Base benchmark:** the original Turkish reasoning questions.
- **Morphosyntactic variants:** controlled variants derived from a subset of the base questions.

This notebook only inspects the datasets.

It does **not**:
- call any API,
- run any language model,
- require a GPU,
- modify the dataset files.

## 1. Setup

The notebook can be used locally or in Google Colab.

If the repository is stored in Google Drive, set `USE_GOOGLE_DRIVE = True` in the next cell.

If the repository was cloned from GitHub, leave it as `False`.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

Python: 3.13.15
pandas: 2.2.3


In [2]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

# Keep this False for the public GitHub version.
# Set it to True when running the project from Google Drive.
USE_GOOGLE_DRIVE = True

DRIVE_PROJECT_PATH = Path(
    "/content/drive/MyDrive/turkish-llm-reasoning"
)


if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        PROJECT_ROOT = DRIVE_PROJECT_PATH

    except ImportError:
        raise RuntimeError(
            "Google Drive mounting is only available in Google Colab."
        )

else:
    # Try to locate the repository automatically.
    current_path = Path.cwd().resolve()
    candidates = [current_path, *current_path.parents]

    PROJECT_ROOT = None

    for candidate in candidates:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            PROJECT_ROOT = candidate
            break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Project root could not be located. "
            "Run the notebook from inside the repository "
            "or enable Google Drive mode."
        )


print("Project root:")
print(PROJECT_ROOT)

Mounted at /content/drive
Project root:
/content/drive/MyDrive/turkish-llm-reasoning


## 2. Load the datasets

The base benchmark and the morphosyntactic variant dataset are stored separately.

The files are loaded without modifying or removing any records.

In [3]:
USE_SMOKE_DATA = True


if USE_SMOKE_DATA:

    BASE_DATA_PATH = (
        PROJECT_ROOT
        / "data"
        / "base"
        / "smoke_base_v1.csv"
    )

    VARIANT_DATA_PATH = (
        PROJECT_ROOT
        / "data"
        / "variants"
        / "smoke_variants_v1.csv"
    )

else:

    BASE_DATA_PATH = (
        PROJECT_ROOT
        / "data"
        / "base"
        / "base_questions.csv"
    )

    VARIANT_DATA_PATH = (
        PROJECT_ROOT
        / "data"
        / "variants"
        / "variants.csv"
    )


if not BASE_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Base dataset not found: {BASE_DATA_PATH}"
    )

if not VARIANT_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Variant dataset not found: {VARIANT_DATA_PATH}"
    )


base_df = pd.read_csv(BASE_DATA_PATH)
variants_df = pd.read_csv(VARIANT_DATA_PATH)


print(f"Base dataset rows: {len(base_df)}")
print(f"Variant dataset rows: {len(variants_df)}")

print("\nBase columns:")
print(list(base_df.columns))

print("\nVariant columns:")
print(list(variants_df.columns))

Base dataset rows: 5
Variant dataset rows: 15

Base columns:
['question_id', 'category', 'difficulty', 'context', 'question', 'gold_evidence', 'gold_answer']

Variant columns:
['variant_id', 'base_question_id', 'category', 'difficulty', 'variant_type', 'morphological_feature', 'context', 'question', 'gold_evidence', 'gold_answer']


## 3. Structural validation

The following checks report:

- missing required values,
- duplicate identifiers,
- structurally complete records.

No automatic corrections are performed.

In [4]:
BASE_REQUIRED_COLUMNS = [
    "question_id",
    "category",
    "difficulty",
    "context",
    "question",
    "gold_evidence",
    "gold_answer",
]

VARIANT_REQUIRED_COLUMNS = [
    "variant_id",
    "base_question_id",
    "category",
    "difficulty",
    "variant_type",
    "morphological_feature",
    "context",
    "question",
    "gold_evidence",
    "gold_answer",
]


def summarize_dataset(
    df,
    id_column,
    required_columns,
    dataset_name,
):
    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing columns: "
            f"{missing_columns}"
        )

    missing_rows = df[required_columns].isna().any(axis=1)

    summary = pd.DataFrame({
        "Metric": [
            "Total rows",
            "Unique IDs",
            "Duplicate IDs",
            "Rows with missing required fields",
            "Structurally complete rows",
        ],
        "Value": [
            len(df),
            df[id_column].nunique(),
            df[id_column].duplicated().sum(),
            missing_rows.sum(),
            (~missing_rows).sum(),
        ],
    })

    print(dataset_name)
    display(summary)

    return missing_rows


base_missing = summarize_dataset(
    base_df,
    "question_id",
    BASE_REQUIRED_COLUMNS,
    "Base dataset",
)

variant_missing = summarize_dataset(
    variants_df,
    "variant_id",
    VARIANT_REQUIRED_COLUMNS,
    "Variant dataset",
)

Base dataset


,Metric,Value
0,Total rows,5
1,Unique IDs,5
2,Duplicate IDs,0
3,Rows with missing required fields,0
4,Structurally complete rows,5


Variant dataset


,Metric,Value
0,Total rows,15
1,Unique IDs,15
2,Duplicate IDs,0
3,Rows with missing required fields,0
4,Structurally complete rows,15


### Historical note

The original base benchmark contains **489 records**.

One record (`TR-1-024`) is missing its `gold_evidence` and `gold_answer`.
The preprocessing used in the original model experiments therefore resulted
in **488 usable questions**.

The incomplete record is preserved in the dataset for provenance rather than
being silently deleted.

In [5]:
if base_missing.any():
    print("Records with missing required fields:")

    display(
        base_df.loc[
            base_missing,
            [
                "question_id",
                "category",
                "difficulty",
                "gold_evidence",
                "gold_answer",
            ],
        ]
    )
else:
    print("No missing required fields found.")

No missing required fields found.


## 4. Dataset distributions

The benchmark contains six reasoning categories and three difficulty levels.

In [6]:
print("BASE DATASET — CATEGORY DISTRIBUTION")

category_counts = (
    base_df["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

display(category_counts)


print("BASE DATASET — DIFFICULTY DISTRIBUTION")

difficulty_counts = (
    base_df["difficulty"]
    .value_counts()
    .rename_axis("difficulty")
    .reset_index(name="count")
)

display(difficulty_counts)


print("BASE DATASET — CATEGORY × DIFFICULTY")

display(
    pd.crosstab(
        base_df["category"],
        base_df["difficulty"],
        margins=True,
    )
)

BASE DATASET — CATEGORY DISTRIBUTION


,category,count
0,temporal_reasoning,1
1,numerical_reasoning,1
2,compositional_reasoning,1
3,causal_reasoning,1
4,logical_inference,1


BASE DATASET — DIFFICULTY DISTRIBUTION


,difficulty,count
0,medium,3
1,easy,2


BASE DATASET — CATEGORY × DIFFICULTY


difficulty,easy,medium,All
category,,,
causal_reasoning,0,1,1
compositional_reasoning,1,0,1
logical_inference,0,1,1
numerical_reasoning,0,1,1
temporal_reasoning,1,0,1
All,2,3,5


## 5. Variant Dataset

The variant dataset contains morphosyntactic modifications derived from a
subset of the original base questions.

Each variant includes a `base_question_id` that links it to its source question.

In [7]:
variant_summary = pd.DataFrame({
    "Metric": [
        "Total variants",
        "Unique base questions",
        "Unique variant types",
        "Reasoning categories",
        "Difficulty levels",
    ],
    "Value": [
        len(variants_df),
        variants_df["base_question_id"].nunique(),
        variants_df["variant_type"].nunique(),
        variants_df["category"].nunique(),
        variants_df["difficulty"].nunique(),
    ],
})

display(variant_summary)


print("VARIANT TYPE DISTRIBUTION")

display(
    variants_df["variant_type"]
    .value_counts()
    .rename_axis("variant_type")
    .reset_index(name="count")
)

,Metric,Value
0,Total variants,15
1,Unique base questions,5
2,Unique variant types,8
3,Reasoning categories,5
4,Difficulty levels,2


VARIANT TYPE DISTRIBUTION


,variant_type,count
0,tense_change,3
1,evidentiality,2
2,conditional_restructuring,2
3,paraphrase,2
4,passive_voice,2
5,negation_form,2
6,temporal_suffix,1
7,quantifier_paraphrase,1


## 6. Base–variant consistency checks

The following checks verify that variants refer to existing base questions
and report metadata inconsistencies.

Detected inconsistencies are only reported; the datasets are not modified.

In [8]:
base_ids = set(
    base_df["question_id"].astype(str)
)

variant_base_ids = set(
    variants_df["base_question_id"].astype(str)
)

unknown_base_ids = sorted(
    variant_base_ids - base_ids
)

print(
    "Variant references not found in base dataset:",
    len(unknown_base_ids),
)

if unknown_base_ids:
    display(
        pd.DataFrame({
            "unknown_base_question_id": unknown_base_ids
        })
    )
else:
    print("✓ Every variant references an existing base question.")

Variant references not found in base dataset: 0
✓ Every variant references an existing base question.


In [9]:
base_metadata = (
    base_df[
        [
            "question_id",
            "category",
            "difficulty",
        ]
    ]
    .rename(
        columns={
            "question_id": "base_question_id",
            "category": "base_category",
            "difficulty": "base_difficulty",
        }
    )
)


variant_check = variants_df.merge(
    base_metadata,
    on="base_question_id",
    how="left",
)


category_mismatches = variant_check[
    variant_check["category"]
    != variant_check["base_category"]
]


print(
    "Variant/base category mismatches:",
    len(category_mismatches),
)


if len(category_mismatches) > 0:
    display(
        category_mismatches[
            [
                "variant_id",
                "base_question_id",
                "category",
                "base_category",
            ]
        ]
    )
else:
    print("✓ Variant categories match their base questions.")

Variant/base category mismatches: 0
✓ Variant categories match their base questions.


## 7. Examples

In [10]:
print("Example base questions")

complete_base = base_df.loc[
    ~base_missing
].copy()

base_examples = (
    complete_base
    .sort_values("question_id")
    .groupby("category", as_index=False)
    .first()
)

display(
    base_examples[
        [
            "question_id",
            "category",
            "difficulty",
            "context",
            "question",
            "gold_answer",
        ]
    ]
)


print("Example variants")

variant_examples = (
    variants_df
    .sort_values("variant_id")
    .groupby("variant_type", as_index=False)
    .first()
)

display(
    variant_examples[
        [
            "variant_id",
            "base_question_id",
            "variant_type",
            "morphological_feature",
            "question",
            "gold_answer",
        ]
    ]
)

Example base questions


,question_id,category,difficulty,context,question,gold_answer
0,CR-2-055,causal_reasoning,medium,"Bir aylık yaz tatiline çıkan Ahmet Bey, evden ...",Bu senaryodaki enerji tüketimine göre; cihazla...,Fişe takılı cihazların bekleme modunda elektri...
1,CM-1-002,compositional_reasoning,easy,Tekno-Global bünyesinde kurulan yeni Ar-Ge eki...,Tekno-Global Ar-Ge ekibinin Takım Kaptanı kim ...,Gizem
2,LR-2-008,logical_inference,medium,Antarktika'daki bir araştırma üssünde görevli ...,"Bu bilgilere göre, Elif'in üssteki görevi nedir?",Biyolog
3,NR-2-009,numerical_reasoning,medium,"""Galaktik Savaşçı"" oyununda, karakterin kullan...","Buna göre, silahın başlangıç enerji kapasitesi...",500
4,TR-1-018,temporal_reasoning,easy,Sabah saat 08.30'da ilk ders zili çalan 3-A sı...,Bu senaryodaki ders programına göre; Türkçe de...,10.00


Example variants


,variant_id,base_question_id,variant_type,morphological_feature,question,gold_answer
0,CM-1-002-v1,CM-1-002,conditional_restructuring,"""sadece X olan ve aynı zamanda Y bilen"" → ""X o...",Tekno-Global Ar-Ge ekibinin Takım Kaptanı kim ...,Gizem
1,CR-2-055-v2,CR-2-055,evidentiality,-miştir → -mış,Bu senaryodaki enerji tüketimine göre; cihazla...,Fişe takılı cihazların bekleme modunda elektri...
2,CM-1-002-v2,CM-1-002,negation_form,"""sadece yazılım mühendisi"" → ""yazılım mühendis...",Tekno-Global Ar-Ge ekibinin Takım Kaptanı kim ...,Gizem
3,CM-1-002-v3,CM-1-002,paraphrase,Cümle düzeyinde yeniden ifade,Tekno-Global Ar-Ge ekibinin Takım Kaptanı kim ...,Gizem
4,CR-2-055-v3,CR-2-055,passive_voice,Etken → Edilgen,Bu senaryodaki enerji tüketimine göre; cihazla...,Fişe takılı cihazların bekleme modunda elektri...
5,NR-2-009-v2,NR-2-009,quantifier_paraphrase,"""%30"" → ""onda üç"", ""%10"" → ""onda bir""","Buna göre, silahın başlangıç enerji kapasitesi...",500
6,TR-1-018-v3,TR-1-018,temporal_suffix,bittiğinde → bitince,Bu senaryodaki ders programına göre; Türkçe de...,10.00
7,CR-2-055-v1,CR-2-055,tense_change,-miştir → -dı,Bu senaryodaki enerji tüketimine göre; cihazla...,Fişe takılı cihazların bekleme modunda elektri...


## 8. Summary

This notebook provides a structural overview of the datasets without
modifying them.

Current dataset structure:

- Base benchmark: **489 total records**
- Structurally complete base records used by the original experiment pipeline: **488**
- Morphosyntactic variants: **257**
- Base questions represented in the variant dataset: **85**
- Variant types: **11**

Model generation and model evaluation are intentionally handled in separate
notebooks.